In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
import matplotlib.pyplot as plt 
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'SACOG Data')
path_main = os.path.join(path_sp, 'Data')

# Git
if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_config  = os.path.join(path_git, 'Python Code', 'Housing', 'config')
if user in ['jchoy', 'AAlAzzawi']:
    path_git     = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_config  = os.path.join(path_git, 'Python Code', 'Housing', 'config')

In [ ]:
path_housing = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'SACOG Housing Dataset')
path_out = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Housing Production')

In [ ]:
# TODO:
# Read in Region_SACOG_Permit_Data.xlsx file by year concatenated together into one dataset
# Then organize the data as described in the crosswalk.xlsx file
# For this script, let's do the indicators:
## Production_1, Production_4, Production_6, Location_1, Location_2a, Location_2b, and Policy_5

# **Housing Data Production (Year Included)**

In [ ]:
year_start = 2001
year_end   = 2022

years_to_import = range(year_start, year_end+1)

list_df = []

for year in years_to_import:
    df_year = pd.read_excel(os.path.join(path_housing, 'Region_SACOG_Permit_Data.xlsx'), sheet_name = str(year))
    df_year['Year'] = year
    list_df.append(df_year)

df_housing = pd.concat(list_df)
df_housing = df_housing.set_index(['County', 'Jurisdiction', 'Year']).reset_index()
df_housing.loc[df_housing['Jurisdiction'].str.contains('COUNTY'), 'Jurisdiction'] = 'UNINCORPORATED'
df_housing

# Export 

# housing_path = os.path.join(path_out, "Housing_Data_Production.xlsx")
# df_housing.to_excel(housing_path, index = False)

# print(f"Data frame exported to {housing_path}")

# **Production_1**

In [ ]:
indicator_name = 'Production_1'

print('Organizing indicator Production_1 by Jurisdictions')
df_prod1_a = df_housing.copy()
df_prod1_a = df_prod1_a[df_prod1_a['County'] != 'Region']
df_prod1_a = df_prod1_a[['County', 'Jurisdiction', 'Year', 'Total']]
display(df_prod1_a.head(5))

print('Organizing indicator Production_1 by Counties')
df_prod1_b = df_housing.copy()
df_prod1_b = df_prod1_b[df_prod1_b['County'] != 'Region']
df_prod1_b = df_prod1_b.groupby(['County', 'Year'], as_index = False)['Total'].agg('sum')
display(df_prod1_b.head(5))

print('Organizing indicator Production_1 by the entire SACOG Region')
df_prod1_c = df_housing.copy()
df_prod1_c = df_prod1_c[df_prod1_c['County'] != 'Region']
df_prod1_c = df_prod1_c.groupby(['Year'], as_index = False)['Total'].agg('sum')
display(df_prod1_c.head(5))

# # Export
# df_prod1_a.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + '_SACOG Housing Permit Data.xlsx'), index = False)
# df_prod1_b.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Counties'      + '_SACOG Housing Permit Data.xlsx'), index = False)
# df_prod1_c.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' MPO'           + '_SACOG Housing Permit Data.xlsx'), index = False)

In [ ]:
df_plot_a = pd.melt(df_prod4_a, id_vars = ['Year', 'County', 'Jurisdiction']) 
fig = px.line(df_plot_a, x='Year', y='value', color='variable', markers=True)

fig.show()

In [ ]:
df_plot = pd.melt(df_prod1_b, id_vars = ['Year', 'County']) 
fig = px.line(df_plot, x='Year', y='value', color='variable', markers=True)

fig.show()

In [ ]:
df_plot = pd.melt(df_prod1_c, id_vars = 'Year') 
fig = px.line(df_plot, x='Year', y='value', color='variable', markers=True)

fig.show()

# **Production_4**

In [ ]:
# Use pd.read_excel to import the Pop_5 jurisdiction data
# Calculate population growth by year using df_pop_5
# Calculate housing growth by year using df_housing "Total" column
# Merge the two files together by county/jurisdiction
# Roll up to Jurisdiction, County, and MPO levels (3 different data frames)
# Make plots

indicator_name = 'Production_4'

path_housing = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'SACOG Housing Dataset')
path_out = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Housing Production')

#Defining Paths
path_housing = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'SACOG Housing Dataset')
pop_1_path = os.path.join(path_main, 'Vibrant and Inclusive Places', 'People and Community', 'Pop and Demographics', 'Pop_1 Total pop')
pop_5_path = os.path.join(path_main, 'Vibrant and Inclusive Places', 'People and Community', 'Pop and Demographics', 'Pop_5 CA Regions')

#Importing Datasets 
df_housing = pd.read_excel(os.path.join(path_out, 'Housing_Data_Production.xlsx'))
df_pop_1 = pd.read_excel(os.path.join(pop_1_path, 'Pop_1_DOF_Counties.xlsx'))
df_pop_5 = pd.read_excel(os.path.join(pop_5_path, 'Pop_5_DOF_Jurisdictions.xlsx'), sheet_name='Data')

In [ ]:
df_housing

In [ ]:
df_pop_5

In [ ]:
df_pop_5 = df_pop_5.sort_values(['County', 'Jurisdiction', 'Year'], ascending=[True,True,True])

#pop_5 for MPO
df_pop_5['Pop_Growth'] = df_pop_5.groupby(['County', 'Jurisdiction'])['Population'].diff()

df_pop_5['County'] = df_pop_5['County'].str.upper() 
df_pop_5['Jurisdiction'] = df_pop_5['Jurisdiction'].str.upper() 

#merging the two files together bby county and jurisdiction
df_merged = pd.merge(df_housing, df_pop_5, on=['County', 'Jurisdiction', 'Year'], how='left')
df_merged = df_merged.rename(columns={'Total': 'Housing_Growth'})

#Jurisdiction level
print('Organizing indicator Production_4 by Jurisdictions')
df_prod4_a = df_merged.copy()
df_prod4_a = df_prod4_a[['County', 'Jurisdiction', 'Year', 'Housing_Growth', 'Pop_Growth']]
display(df_prod4_a.head(5))

#County level 
print('Organizing indicator Production_4 by Counties')
df_prod4_b = df_merged.copy()
df_prod4_b = df_prod4_b.groupby(['County', 'Year'], as_index=False).agg({
    # 'Total': 'sum',
    'Housing_Growth': 'sum',
    # 'Population': 'sum',
    'Pop_Growth': 'sum'
})
display(df_prod4_b.head(5))

#MPO level
#Jurisdiction level
print('Organizing indicator Production_4 by MPO')
df_prod4_c = df_merged.copy()
df_prod4_c = df_prod4_c.groupby(['Year'], as_index=False).agg({
    # 'Total': 'sum',
    'Housing_Growth': 'sum',
    # 'Population': 'sum',
    'Pop_Growth': 'sum'
})
display(df_prod4_c.head(5))

# #Export
df_prod4_a.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + '_SACOG Housing and Population Data.xlsx'), index=False)
df_prod4_b.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Counties' + '_SACOG Housing and Population Data.xlsx'), index=False)
df_prod4_c.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' MPO' + '_SACOG Housing and Population Data.xlsx'), index=False)

print(f"Data frames exported to {path_out}")

In [ ]:
df_plot_a = pd.melt(df_prod4_a, id_vars = ['Year', 'County', 'Jurisdiction']) 
fig = px.line(df_plot, x='Year', y='value', color='variable', markers=True)

fig.show()

In [ ]:
df_plot_b = pd.melt(df_prod4_b, id_vars = ['Year', 'County']) 
fig = px.line(df_plot, x='Year', y='value', color='variable', markers=True)

fig.show()

In [ ]:
df_plot_c = pd.melt(df_prod4_c, id_vars = 'Year') 
fig = px.line(df_plot, x='Year', y='value', color='variable', markers=True)

fig.show()

# **Production_6**

In [ ]:
path_housing = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'SACOG Housing Dataset')
path_out = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Housing Production')

df_housing = pd.read_excel(os.path.join(path_out, 'Housing_Data_Production.xlsx'))

In [ ]:
df_housing

In [ ]:
gb_jurisdiction = df_housing.groupby(['County', 'Jurisdiction', 'Year'], as_index=False)
gb_county = df_housing.groupby(['County', 'Year'], as_index=False)
gb_mpo = df_housing.groupby(['Year'], as_index=False)

In [ ]:
indicator_name = "Production_6"

df_prod_6 = df_housing[['County', 'Jurisdiction', 'Year', 'MF_total', 'MF_2to4', 'MF_5plus', 'SF_RR', 'SF_SFLL']]

metrics = ['SF_total', 'MF_total', 'MF_2to4', 'MF_5plus', 'SF_RR', 'SF_SFLL']

#Jurisdiction level
print('Organizing indicator Production_6 by Jurisdictions')
df_prod6_a = gb_jurisdiction[metrics].sum() 
display(df_prod6_a.head(5))

#County level 
print('Organizing indicator Production_6 by Counties')
df_prod6_b = gb_county[metrics].sum()
display(df_prod6_b.head(5))

#MPO level
#Jurisdiction level
print('Organizing indicator Production_6 by MPO')
df_prod6_c = gb_mpo[metrics].sum()
display(df_prod6_c.head(5))

# #Export
df_prod6_a.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + '_SACOG Housing and Population Data.xlsx'), index=False)
df_prod6_b.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Counties' + '_SACOG Housing and Population Data.xlsx'), index=False)
df_prod6_c.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' MPO' + '_SACOG Housing and Population Data.xlsx'), index=False)

print(f"Data frames exported to {path_out}")

In [ ]:
df_plot_a = df_prod6_a.drop(['MF_total', 'SF_total'], axis = 1)
df_plot_a = pd.melt(df_plot_a, id_vars = ['Year', 'County', 'Jurisdiction']) 
fig = px.line(df_plot, x='Year', y='value', color='variable', markers=True)

fig.show()

In [ ]:
df_plot_b = df_prod6_b.drop(['MF_total', 'SF_total'], axis = 1)
df_plot_b = pd.melt(df_plot_b, id_vars = ['Year', 'County']) 
fig = px.line(df_plot, x='Year', y='value', color='variable', markers=True)

fig.show()

In [ ]:
df_plot = df_prod6_c.drop(['MF_total', 'SF_total'], axis = 1)
df_plot = pd.melt(df_plot, id_vars = 'Year') 
fig = px.line(df_plot, x='Year', y='value', color='variable', markers=True)

fig.show()

In [ ]:
indicator_name = "Production_6"

df_prod_6 = df_housing[['County', 'Jurisdiction', 'Year', 'MF_total', 'MF_2to4', 'MF_5plus', 'SF_RR', 'SF_SFLL']]

metrics = ['MF_total', 'MF_2to4', 'MF_5plus', 'SF_RR', 'SF_SFLL']

#Jurisdiction level
print('Organizing indicator Production_6 by Jurisdictions')
df_prod6_a = df_prod_6.copy()
df_prod6_a = df_prod6_a[['County', 'Jurisdiction', 'Year'] + metrics]
display(df_prod6_a.head(5))

#County level 
print('Organizing indicator Production_6 by Counties')
df_prod6_b = df_prod_6.copy()
df_prod6_b = df_prod6_b.groupby(['County', 'Year'], as_index=False)[metrics].sum()
display(df_prod6_b.head(5))

#MPO level
#Jurisdiction level
print('Organizing indicator Production_6 by MPO')
df_prod6_c = df_prod_6.copy()
df_prod6_c = df_prod6_c.groupby(['Year'], as_index=False)[metrics].sum()
display(df_prod6_c.head(5))

#Export
# df_prod6_a.to_excel(os.path.join(path_out, indicator_name, indicator_name + 'Jurisdictions' + '_SACOG Housing and Population Data.xlsx'), index=False)
# df_prod6_b.to_excel(os.path.join(path_out, indicator_name, indicator_name + 'Counties' + '_SACOG Housing and Population Data.xlsx'), index=False)
# df_prod6_c.to_excel(os.path.join(path_out, indicator_name, indicator_name + 'MPO' + '_SACOG Housing and Population Data.xlsx'), index=False)

# print(f"Data frames exported to {path_out}")

# **Location_1**

In [ ]:
gb_jurisdiction = df_housing.groupby(['County', 'Jurisdiction', 'Year'], as_index=False)
gb_county = df_housing.groupby(['County', 'Year'], as_index=False)
gb_mpo = df_housing.groupby(['Year'], as_index=False)

In [ ]:
indicator_name = "Location_1"

df_loc_1 = df_housing[['County', 'Jurisdiction', 'Year', 'ADU_DEMO', 'COMTYP_CC', 'COMTYP_EC', 'COMTYP_DC']]

metrics = ['ADU_DEMO', 'COMTYP_CC', 'COMTYP_EC', 'COMTYP_DC']

#Jurisdiction level
print('Organizing indicator Location_1 by Jurisdictions')
df_loc1_a = gb_jurisdiction[metrics].sum() 
display(df_loc1_a.head(5))

#County level 
print('Organizing indicator Location_1 by Counties')
df_loc1_b = gb_county[metrics].sum()
display(df_loc1_b.head(5))

#MPO level
#Jurisdiction level
print('Organizing indicator Location_1 by MPO')
df_loc1_c = gb_mpo[metrics].sum()
display(df_loc1_c.head(5))

#Export
df_loc1_a.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + '_SACOG Housing and Population Data.xlsx'), index=False)
df_loc1_b.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Counties' + '_SACOG Housing and Population Data.xlsx'), index=False)
df_loc1_c.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' MPO' + '_SACOG Housing and Population Data.xlsx'), index=False)

print(f"Data frames exported to {path_out}")



# **Location_2a**

In [ ]:
path_housing = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'SACOG Housing Dataset')
path_out = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Housing Production')

df_housing = pd.read_excel(os.path.join(path_out, 'Housing_Data_Production.xlsx'))


In [ ]:
indicator_name = "Location_2a"

# df_loc_2a = df_housing[['County', 'Jurisdiction', 'Year', 'SF_total']]

metrics = ['SF_total']

#Jurisdiction level
print('Organizing indicator Location_2a by Jurisdictions')
df_loc2a_a = gb_jurisdiction[metrics].sum() 
display(df_loc2a_a.head(5))

#County level 
print('Organizing indicator Location_2a by Counties')
df_loc2a_b = gb_county[metrics].sum()
display(df_loc2a_b.head(5))

#MPO level
#Jurisdiction level
print('Organizing indicator Location_2a by MPO')
df_loc2a_c = gb_mpo[metrics].sum()
display(df_loc2a_c.head(5))

#Export
# df_loc2a_a.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + '_SACOG Housing and Population Data.xlsx'), index=False)
# df_loc2a_b.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Counties' + '_SACOG Housing and Population Data.xlsx'), index=False)
# df_loc2a_c.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' MPO' + '_SACOG Housing and Population Data.xlsx'), index=False)

# **Location_2b**

In [ ]:
path_housing = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'SACOG Housing Dataset')
path_out = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Housing Production')

df_housing = pd.read_excel(os.path.join(path_out, 'Housing_Data_Production.xlsx'))

In [ ]:
indicator_name = "Location_2b"

df_loc_2b = df_housing[['County', 'Jurisdiction', 'Year', 'SF_total']]

metrics_1 = ['GRZ_SF', 'GRZ_MF2t4']
metrics_2 = ['GRZ_SFSL', 'GRZ_SFLL']
metrics_3 = ['GRZ_SF_NO', 'GRZ_MF2t4_NO']
metrics_4 = ['GRZ_SFSL_NO', 'GRZ_SFLL_NO']

#Jurisdiction level for Green Zone region 
print('Organizing indicator Location_2a by Jurisdictions for Green Zone region')
df_loc2b_a = gb_jurisdiction[metrics_1 + metrics_2].sum() 
display(df_loc2b_a.head(5))

#County level for Green Zone Region
print('Organizing indicator Location_2a by Counties for Green Zone region')
df_loc2b_b = gb_county[metrics_1 + metrics_2].sum()
display(df_loc2b_b.head(5))

#MPO level for Green Zone Region 
print('Organizing indicator Location_2a by MPO for Green Zone region')
df_loc2b_c = gb_mpo[metrics_1 + metrics_2].sum()
display(df_loc2b_c.head(5))


#Jurisdiction level for rest region 
print('Organizing indicator Location_2a by Jurisdictions for rest of region')
df_loc2b_a1 = gb_jurisdiction[metrics_3 + metrics_4].sum() 
display(df_loc2b_a1.head(5))

#County level for Green Zone Region
print('Organizing indicator Location_2a by Counties for rest of region')
df_loc2b_b1 = gb_county[metrics_3 + metrics_4].sum()
display(df_loc2b_b1.head(5))

#MPO level for Green Zone Region 
print('Organizing indicator Location_2a by MPO for rest of region')
df_loc2b_c1 = gb_mpo[metrics_3 + metrics_4].sum()
display(df_loc2b_c1.head(5))

#Export
# df_loc2b_a.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + '_SACOG Housing and Population Data.xlsx'), index=False)
# df_loc2b_b.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Counties' + '_SACOG Housing and Population Data.xlsx'), index=False)
# df_loc2b_c.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' MPO' + '_SACOG Housing and Population Data.xlsx'), index=False)

# df_loc2b_a1.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + '_SACOG Housing and Population Data.xlsx'), index=False)
# df_loc2b_b1.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Counties' + '_SACOG Housing and Population Data.xlsx'), index=False)
# df_loc2b_c1.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' MPO' + '_SACOG Housing and Population Data.xlsx'), index=False)


# **Policy_5**

In [ ]:
path_housing = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'SACOG Housing Dataset')
path_out = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Housing Production')

df_housing = pd.read_excel(os.path.join(path_out, 'Housing_Data_Production.xlsx'))

In [ ]:
gb_jurisdiction = df_housing.groupby(['County', 'Jurisdiction', 'Year'], as_index=False)
gb_county = df_housing.groupby(['County', 'Year'], as_index=False)
gb_mpo = df_housing.groupby(['Year'], as_index=False)

mtp_projection = [110, 600, 2100]

In [ ]:
indicator_name = 'Policy_5'

metrics = ['SF_total', 'SF_RR', 'SF_SFLL', 'SF_SFSL']

#Jurisdiction level
print('Organizing indicator Policy_5 by Jurisdictions')
df_policy5_a = gb_jurisdiction[metrics].sum() 

df_policy5_a['MTP_RR'] = mtp_projection[0]
df_policy5_a['MTP_SFLL'] = mtp_projection[1]
df_policy5_a['MTP_SFSL'] = mtp_projection[2]

display(df_policy5_a.head(5))

#County level 
print('Organizing indicator Policy_5 by Counties')
df_policy5_b = gb_county[metrics].sum()

df_policy5_b['MTP_RR'] = mtp_projection[0]
df_policy5_b['MTP_SFLL'] = mtp_projection[1]
df_policy5_b['MTP_SFSL'] = mtp_projection[2]

display(df_policy5_b.head(5))

#MPO level
#Jurisdiction level
print('Organizing indicator Policy_5 by MPO')
df_policy5_c = gb_mpo[metrics].sum()

df_policy5_c['MTP_RR'] = mtp_projection[0]
df_policy5_c['MTP_SFLL'] = mtp_projection[1]
df_policy5_c['MTP_SFSL'] = mtp_projection[2]

display(df_policy5_c.head(5))

In [ ]:
# forecast 

forecast_1 = (2001, 2007)
forecast_2 = (2008, 2011)
forecast_3 = (2012, 2015)
forecast_4 = (2016, 2019)
forecast_5 = (2020, 2024)

forecasts = [forecast_1, forecast_2, forecast_3, forecast_4, forecast_5]

for policy in [df_policy5_a, df_policy5_b, df_policy5_c]:




# #Export 
# df_policy5_a.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + '_SACOG Housing and Population Data.xlsx'), index=False)
# df_policy5_b.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Counties' + '_SACOG Housing and Population Data.xlsx'), index=False)
# df_policy5_c.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' MPO' + '_SACOG Housing and Population Data.xlsx'), index=False)

    